# 02 - Modelado supervisado

Este notebook implementa y entrena modelos supervisados base para dos enfoques complementarios:

1. **Clasificación**
   - Objetivo: predecir si un cliente abandona o no el servicio.
   - Target: `abandono`.
   - Modelos: `LogisticRegression`, `DecisionTreeClassifier` y `SVM`.

2. **Regresión**
   - Objetivo: estimar una variable numérica relevante del cliente.
   - Target: `gasto_mensual`.
   - Modelos: `LinearRegression` y `DecisionTreeRegressor`.

La clasificación aborda el riesgo de abandono. La regresión complementa ese análisis al estimar una dimensión económica del cliente.

El resultado persistente de este notebook no es una nueva tabla de datos, sino modelos/pipelines entrenados, métricas base, particiones train/test y metadata del modelado.

## 1. Importación de librerías

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data_preprocessing import (
    crear_preprocesador,
    guardar_indices_split,
    crear_pipelines_clasificacion,
    crear_pipelines_regresion
)

from src.model_training import (
    obtener_modelos_clasificacion,
    obtener_modelos_regresion,
    entrenar_modelo,
    guardar_modelos_entrenados
)

from src.model_evaluation import obtener_score_clasificacion, guardar_metricas

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted

## 2. Configuración general

Se define una configuración única para asegurar reproducibilidad en particiones, modelos y rutas de salida.

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "data_transformada.csv",
    PROJECT_ROOT / "data_transformada.csv",
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró data_transformada.csv. "
        "Ubica el archivo en data/processed/data_transformada.csv o en la raíz del proyecto."
    )

METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
SPLITS_DIR = PROJECT_ROOT / "results" / "splits"
MODELS_DIR = PROJECT_ROOT / "models" / "trained_models"

for directory in [METRICS_DIR, SPLITS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Proyecto raíz:", PROJECT_ROOT)
print("Dataset de entrada:", DATA_PATH)
print("Métricas:", METRICS_DIR)
print("Splits:", SPLITS_DIR)
print("Modelos entrenados:", MODELS_DIR)

Proyecto raíz: /datasets/_deepnote_work
Dataset de entrada: /datasets/_deepnote_work/data/processed/data_transformada.csv
Métricas: /datasets/_deepnote_work/results/metrics
Splits: /datasets/_deepnote_work/results/splits
Modelos entrenados: /datasets/_deepnote_work/models/trained_models


## 3. Carga del dataset

Se usa la base procesada disponible en `data/processed/data_transformada.csv`.

Este archivo permanece sin modificación; las operaciones de preprocesamiento se aplican dentro de los pipelines entrenados.

In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Dimensiones del dataset: {df.shape}")
display(df.head())

Dimensiones del dataset: (20000, 38)


,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,ratio_deuda_ingreso,ratio_gasto_ingreso,edad,antiguedad_meses,frecuencia_compra,ultima_compra_dias,...,canal_registro_Tienda,canal_registro_Web,tiene_tarjeta_credito,hora_sin,hora_cos,mes_sin,mes_cos,dia_semana_sin,dia_semana_cos,abandono
0,0.532426,0.933822,0.566762,-1.627731,-0.084464,0.112246,0.979150,1.193186,-0.724598,1.653219,...,1.0,0.0,1.0,-0.500000,8.660254e-01,-0.866025,5.000000e-01,0.000000,1.000000,1
1,1.863965,-0.651529,-0.556596,-0.286833,-1.157946,-1.258407,0.139166,0.140295,-0.358339,1.186639,...,0.0,0.0,1.0,0.500000,-8.660254e-01,-0.866025,-5.000000e-01,0.781831,0.623490,0
2,-0.002148,-0.100490,1.864029,1.840545,-0.179591,-0.172190,-0.028831,-1.731512,0.374180,0.472486,...,0.0,0.0,1.0,1.000000,6.123234e-17,0.500000,-8.660254e-01,0.433884,-0.900969,0
3,-1.646975,0.127203,1.314972,-1.769889,2.442661,1.676920,0.307163,-1.643771,-1.273987,-0.165490,...,0.0,0.0,1.0,-0.866025,-5.000000e-01,0.866025,-5.000000e-01,-0.781831,0.623490,1
4,-0.287862,0.683531,-0.533327,-1.484479,-0.402574,0.497931,-0.980813,-1.351302,-0.907728,0.958110,...,0.0,1.0,1.0,0.866025,-5.000000e-01,1.000000,6.123234e-17,0.781831,0.623490,1


## 4. Verificación mínima del dataset

El análisis exploratorio completo pertenece al notebook `01_exploratory_analysis`.

Aquí solo se valida que la base cargada sea utilizable para modelado.

In [4]:
print("Tipos de datos:")
display(df.dtypes.value_counts())

print("\nColumnas con más nulos:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicados:", df.duplicated().sum())

Tipos de datos:


float64    37
int64       1
Name: count, dtype: int64


Columnas con más nulos:


ingreso_mensual          0
canal_registro_Tienda    0
uso_app_Alto             0
uso_app_Bajo             0
uso_app_Medio            0
tipo_plan_Basico         0
tipo_plan_Estandar       0
tipo_plan_Premium        0
canal_registro_App       0
canal_registro_Web       0
gasto_mensual            0
tiene_tarjeta_credito    0
hora_sin                 0
hora_cos                 0
mes_sin                  0
dtype: int64


Duplicados: 0


## 5. Definición de problemas supervisados

Se mantienen dos problemas independientes:

- `abandono` como target de clasificación.
- `gasto_mensual` como target de regresión.

La regresión no se aplica sobre `abandono`, porque `abandono` representa una clase. Para regresión se usa una variable numérica con interpretación económica.

In [5]:
TARGET_CLASIFICACION = "abandono"
TARGET_REGRESION = "gasto_mensual"

for target in [TARGET_CLASIFICACION, TARGET_REGRESION]:
    if target not in df.columns:
        raise ValueError(f"No se encontró la columna target requerida: {target}")

print("Target de clasificación:", TARGET_CLASIFICACION)
print("Target de regresión:", TARGET_REGRESION)

Target de clasificación: abandono
Target de regresión: gasto_mensual


## 7. Preparación del problema de clasificación

In [6]:
df_clf = df.dropna(subset=[TARGET_CLASIFICACION]).copy()

columnas_excluir_clf = [TARGET_CLASIFICACION]
if "id_cliente" in df_clf.columns:
    columnas_excluir_clf.append("id_cliente")

X_clf = df_clf.drop(columns=columnas_excluir_clf)
y_clf_original = df_clf[TARGET_CLASIFICACION]

label_encoder_clf = LabelEncoder()
y_clf = pd.Series(
    label_encoder_clf.fit_transform(y_clf_original),
    index=df_clf.index,
    name=TARGET_CLASIFICACION
)

df_label_mapping = pd.DataFrame({
    "clase_original": label_encoder_clf.classes_,
    "clase_codificada": label_encoder_clf.transform(label_encoder_clf.classes_)
})

label_mapping_path = METRICS_DIR / "classification_label_mapping.csv"
df_label_mapping.to_csv(label_mapping_path, index=False)

print("Columnas excluidas en clasificación:", columnas_excluir_clf)
print("Dimensiones X_clf:", X_clf.shape)
print("Dimensiones y_clf:", y_clf.shape)
print("\nMapeo del target de clasificación:")
display(df_label_mapping)

Columnas excluidas en clasificación: ['abandono']
Dimensiones X_clf: (20000, 37)
Dimensiones y_clf: (20000,)

Mapeo del target de clasificación:


,clase_original,clase_codificada
0,0,0
1,1,1


In [7]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

guardar_indices_split(X_train_clf.index, SPLITS_DIR / "classification_train_indices.csv")
guardar_indices_split(X_test_clf.index, SPLITS_DIR / "classification_test_indices.csv")

print("Train clasificación:", X_train_clf.shape)
print("Test clasificación:", X_test_clf.shape)

print("\nDistribución del target en train:")
display(y_train_clf.value_counts(normalize=True).rename("proporcion"))

print("\nDistribución del target en test:")
display(y_test_clf.value_counts(normalize=True).rename("proporcion"))

Train clasificación: (16000, 37)
Test clasificación: (4000, 37)

Distribución del target en train:


abandono
0    0.60325
1    0.39675
Name: proporcion, dtype: float64


Distribución del target en test:


abandono
0    0.60325
1    0.39675
Name: proporcion, dtype: float64

## 8. Pipelines y modelos base de clasificación

Se usan modelos base desde `src/model_training.py`.

Cada modelo se integra en un pipeline con el preprocesador correspondiente:

- Escalamiento para `LogisticRegression` y `SVM`.
- Preprocesamiento sin escalamiento para `DecisionTreeClassifier`.

In [8]:
preprocesador_clf_escalado, grupos_clf = crear_preprocesador(X_train_clf, escalar=True)
preprocesador_clf_arbol, _ = crear_preprocesador(X_train_clf, escalar=False)

print("Grupos de columnas usados en clasificación:")
for grupo, columnas in grupos_clf.items():
    print(f"{grupo}: {columnas}")

modelos_clasificacion = crear_pipelines_clasificacion(
    preprocesador_escalado=preprocesador_clf_escalado,
    preprocesador_arbol=preprocesador_clf_arbol,
    random_state=RANDOM_STATE,
)

print("Modelos de clasificación:")
print(list(modelos_clasificacion.keys()))

Grupos de columnas usados en clasificación:
outliers: ['ingreso_mensual', 'gasto_mensual', 'deuda_total', 'score_crediticio', 'ratio_deuda_ingreso', 'ratio_gasto_ingreso']
numericas: ['edad', 'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos', 'anio', 'genero_Femenino', 'genero_Masculino', 'genero_Otro', 'region_Centro', 'region_Norte', 'region_Sur', 'estado_civil_Casado', 'estado_civil_Divorciado', 'estado_civil_Soltero', 'uso_app_Alto', 'uso_app_Bajo', 'uso_app_Medio', 'tipo_plan_Basico', 'tipo_plan_Estandar', 'tipo_plan_Premium', 'canal_registro_App', 'canal_registro_Tienda', 'canal_registro_Web']
categoricas: []
binarias: ['tiene_tarjeta_credito']
ciclicas: ['hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'dia_semana_sin', 'dia_semana_cos']
Modelos de clasificación:
['Logistic Regression', 'Decision Tree Classifier', 'SVM']


## 9. Entrenamiento base de clasificación

Se entrenan los pipelines completos.

El preprocesamiento aprendido queda contenido dentro de cada pipeline entrenado.

In [9]:
modelos_clasificacion_entrenados = {}
resultados_clasificacion = []

for nombre, pipeline in modelos_clasificacion.items():
    print(f"Entrenando: {nombre}")
    modelo_entrenado = entrenar_modelo(pipeline, X_train_clf, y_train_clf)
    modelos_clasificacion_entrenados[nombre] = modelo_entrenado

    y_pred = modelo_entrenado.predict(X_test_clf)
    y_score = obtener_score_clasificacion(modelo_entrenado, X_test_clf)
    roc_auc = (
        roc_auc_score(y_test_clf, y_score)
        if y_score is not None and len(np.unique(y_test_clf)) == 2
        else np.nan
    )

    resultados_clasificacion.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_clf, y_pred),
        "precision": precision_score(y_test_clf, y_pred, zero_division=0),
        "recall": recall_score(y_test_clf, y_pred, zero_division=0),
        "f1_score": f1_score(y_test_clf, y_pred, zero_division=0),
        "roc_auc": roc_auc
    })

df_resultados_clasificacion_base = pd.DataFrame(resultados_clasificacion)
display(df_resultados_clasificacion_base)

Entrenando: Logistic Regression
Entrenando: Decision Tree Classifier
Entrenando: SVM


,modelo,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.65500,0.604230,0.378072,0.465116,0.688220
1,Decision Tree Classifier,0.55875,0.447212,0.475110,0.460739,0.544435
2,SVM,0.64450,0.602230,0.306238,0.406015,0.656956


## 10. Preparación del problema de regresión

Se usa `gasto_mensual` como variable objetivo.

Para evitar fuga de información, se excluyen variables no predictoras o directamente asociadas al target de regresión.

In [10]:
df_reg = df.dropna(subset=[TARGET_REGRESION]).copy()

columnas_excluir_reg = [TARGET_REGRESION]

for col in ["id_cliente", TARGET_CLASIFICACION, "ratio_gasto_ingreso"]:
    if col in df_reg.columns:
        columnas_excluir_reg.append(col)

X_reg = df_reg.drop(columns=columnas_excluir_reg)
y_reg = pd.to_numeric(df_reg[TARGET_REGRESION], errors="coerce")

filas_validas_reg = y_reg.notna()
X_reg = X_reg.loc[filas_validas_reg]
y_reg = y_reg.loc[filas_validas_reg]

print("Columnas excluidas en regresión:", columnas_excluir_reg)
print("Dimensiones X_reg:", X_reg.shape)
print("Dimensiones y_reg:", y_reg.shape)

Columnas excluidas en regresión: ['gasto_mensual', 'abandono', 'ratio_gasto_ingreso']
Dimensiones X_reg: (20000, 35)
Dimensiones y_reg: (20000,)


In [11]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

guardar_indices_split(X_train_reg.index, SPLITS_DIR / "regression_train_indices.csv")
guardar_indices_split(X_test_reg.index, SPLITS_DIR / "regression_test_indices.csv")

print("Train regresión:", X_train_reg.shape)
print("Test regresión:", X_test_reg.shape)

Train regresión: (16000, 35)
Test regresión: (4000, 35)


## 11. Pipelines y modelos base de regresión

Se usan modelos base desde `src/model_training.py`.

- `LinearRegression` usa preprocesamiento con escalamiento.
- `DecisionTreeRegressor` usa preprocesamiento sin escalamiento.

In [12]:
preprocesador_reg_escalado, grupos_reg = crear_preprocesador(X_train_reg, escalar=True)
preprocesador_reg_arbol, _ = crear_preprocesador(X_train_reg, escalar=False)

print("Grupos de columnas usados en regresión:")
for grupo, columnas in grupos_reg.items():
    print(f"{grupo}: {columnas}")

modelos_regresion = crear_pipelines_regresion(
    preprocesador_escalado=preprocesador_reg_escalado,
    preprocesador_arbol=preprocesador_reg_arbol,
    random_state=RANDOM_STATE,
)

print("Modelos de regresión:")
print(list(modelos_regresion.keys()))

Grupos de columnas usados en regresión:
outliers: ['ingreso_mensual', 'deuda_total', 'score_crediticio', 'ratio_deuda_ingreso']
numericas: ['edad', 'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos', 'anio', 'genero_Femenino', 'genero_Masculino', 'genero_Otro', 'region_Centro', 'region_Norte', 'region_Sur', 'estado_civil_Casado', 'estado_civil_Divorciado', 'estado_civil_Soltero', 'uso_app_Alto', 'uso_app_Bajo', 'uso_app_Medio', 'tipo_plan_Basico', 'tipo_plan_Estandar', 'tipo_plan_Premium', 'canal_registro_App', 'canal_registro_Tienda', 'canal_registro_Web']
categoricas: []
binarias: ['tiene_tarjeta_credito']
ciclicas: ['hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'dia_semana_sin', 'dia_semana_cos']
Modelos de regresión:
['Linear Regression', 'Decision Tree Regressor']


## 12. Entrenamiento base de regresión

Se entrenan los pipelines completos y se calculan métricas preliminares.

In [13]:
modelos_regresion_entrenados = {}
resultados_regresion = []

for nombre, pipeline in modelos_regresion.items():
    print(f"Entrenando: {nombre}")
    modelo_entrenado = entrenar_modelo(pipeline, X_train_reg, y_train_reg)
    modelos_regresion_entrenados[nombre] = modelo_entrenado

    y_pred = modelo_entrenado.predict(X_test_reg)

    resultados_regresion.append({
        "modelo": nombre,
        "mae": mean_absolute_error(y_test_reg, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test_reg, y_pred)),
        "r2": r2_score(y_test_reg, y_pred)
    })

df_resultados_regresion_base = pd.DataFrame(resultados_regresion)
display(df_resultados_regresion_base)

Entrenando: Linear Regression
Entrenando: Decision Tree Regressor


,modelo,mae,rmse,r2
0,Linear Regression,0.816198,1.006341,-0.001883
1,Decision Tree Regressor,1.152062,1.426339,-1.012671


## 13. Guardado de artefactos del modelado base

Se guardan métricas, splits y pipelines entrenados.

Estos archivos son la salida persistente del notebook.

In [14]:
classification_metrics_path = METRICS_DIR / "classification_base_metrics.csv"
regression_metrics_path = METRICS_DIR / "regression_base_metrics.csv"

# Guardar métricas
guardar_metricas(df_resultados_clasificacion_base, classification_metrics_path)
guardar_metricas(df_resultados_regresion_base, regression_metrics_path)

# Guardar modelos entrenados y construir manifiesto
manifest_registros = []
manifest_registros.extend(
    guardar_modelos_entrenados(
        modelos_clasificacion_entrenados,
        tipo_modelo="classification",
        models_dir=MODELS_DIR,
        project_root=PROJECT_ROOT,
        target=TARGET_CLASIFICACION
    )
)
manifest_registros.extend(
    guardar_modelos_entrenados(
        modelos_regresion_entrenados,
        tipo_modelo="regression",
        models_dir=MODELS_DIR,
        project_root=PROJECT_ROOT,
        target=TARGET_REGRESION
    )
)

# Crear y guardar el manifiesto de modelos
df_model_manifest = pd.DataFrame(manifest_registros)
manifest_path = MODELS_DIR / "trained_models_manifest.csv"
df_model_manifest.to_csv(manifest_path, index=False)

# Construcción de metadata del modelado
try:
    data_path_for_metadata = str(DATA_PATH.relative_to(PROJECT_ROOT))
except ValueError:
    data_path_for_metadata = str(DATA_PATH)

metadata = {
    "data_path": data_path_for_metadata,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_clasificacion": TARGET_CLASIFICACION,
    "target_regresion": TARGET_REGRESION,
    "classification_train_indices": "results/splits/classification_train_indices.csv",
    "classification_test_indices": "results/splits/classification_test_indices.csv",
    "regression_train_indices": "results/splits/regression_train_indices.csv",
    "regression_test_indices": "results/splits/regression_test_indices.csv",
    "classification_metrics": "results/metrics/classification_base_metrics.csv",
    "regression_metrics": "results/metrics/regression_base_metrics.csv",
    "classification_label_mapping": "results/metrics/classification_label_mapping.csv",
    "trained_models_manifest": "models/trained_models/trained_models_manifest.csv"
}

# Guardar metadata
metadata_path = MODELS_DIR / "modeling_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=4)

print("Métricas guardadas:")
print("-", classification_metrics_path)
print("-", regression_metrics_path)

print("\nModelos entrenados guardados:")
from IPython.display import display
display(df_model_manifest)

print("\nMetadata guardada en:", metadata_path)

Métricas guardadas:
- /datasets/_deepnote_work/results/metrics/classification_base_metrics.csv
- /datasets/_deepnote_work/results/metrics/regression_base_metrics.csv

Modelos entrenados guardados:


,tipo_modelo,modelo,ruta_modelo,target
0,classification,Logistic Regression,models/trained_models/classification_logistic_...,abandono
1,classification,Decision Tree Classifier,models/trained_models/classification_decision_...,abandono
2,classification,SVM,models/trained_models/classification_svm_pipel...,abandono
3,regression,Linear Regression,models/trained_models/regression_linear_regres...,gasto_mensual
4,regression,Decision Tree Regressor,models/trained_models/regression_decision_tree...,gasto_mensual



Metadata guardada en: /datasets/_deepnote_work/models/trained_models/modeling_metadata.json


## 14. Comprobaciones finales

Se verifica que los productos persistentes del notebook existen y que los modelos entrenados están ajustados.

In [15]:
checks = []

def registrar(nombre, condicion, detalle_ok="", detalle_error=""):
    checks.append({
        "comprobacion": nombre,
        "estado": "OK" if condicion else "REVISAR",
        "detalle": detalle_ok if condicion else detalle_error
    })

archivos_requeridos = [
    classification_metrics_path,
    regression_metrics_path,
    manifest_path,
    metadata_path,
    SPLITS_DIR / "classification_train_indices.csv",
    SPLITS_DIR / "classification_test_indices.csv",
    SPLITS_DIR / "regression_train_indices.csv",
    SPLITS_DIR / "regression_test_indices.csv",
    label_mapping_path,
]

for ruta in archivos_requeridos:
    registrar(
        f"Existe archivo {ruta.relative_to(PROJECT_ROOT)}",
        ruta.exists(),
        "Archivo encontrado.",
        "Archivo no encontrado."
    )

for ruta_modelo in df_model_manifest["ruta_modelo"]:
    ruta = PROJECT_ROOT / ruta_modelo
    registrar(
        f"Existe modelo {ruta_modelo}",
        ruta.exists(),
        "Modelo serializado encontrado.",
        "Modelo serializado no encontrado."
    )

for nombre, modelo in {**modelos_clasificacion_entrenados, **modelos_regresion_entrenados}.items():
    try:
        check_is_fitted(modelo)
        ajustado = True
    except Exception:
        ajustado = False

    registrar(
        f"Modelo entrenado en memoria: {nombre}",
        ajustado,
        "Modelo ajustado.",
        "Modelo no ajustado."
    )

registrar(
    "Target de clasificación no está en X_clf",
    TARGET_CLASIFICACION not in X_clf.columns,
    "Correcto.",
    "El target de clasificación aparece como predictor."
)

registrar(
    "Target de regresión no está en X_reg",
    TARGET_REGRESION not in X_reg.columns,
    "Correcto.",
    "El target de regresión aparece como predictor."
)

registrar(
    "abandono no está en X_reg",
    TARGET_CLASIFICACION not in X_reg.columns,
    "Correcto.",
    "La variable de clasificación aparece en predictores de regresión."
)

df_checks = pd.DataFrame(checks)
display(df_checks)

pendientes = (df_checks["estado"] == "REVISAR").sum()
print(f"Comprobaciones pendientes de revisión: {pendientes}")

,comprobacion,estado,detalle
0,Existe archivo results/metrics/classification_...,OK,Archivo encontrado.
1,Existe archivo results/metrics/regression_base...,OK,Archivo encontrado.
2,Existe archivo models/trained_models/trained_m...,OK,Archivo encontrado.
3,Existe archivo models/trained_models/modeling_...,OK,Archivo encontrado.
4,Existe archivo results/splits/classification_t...,OK,Archivo encontrado.
5,Existe archivo results/splits/classification_t...,OK,Archivo encontrado.
6,Existe archivo results/splits/regression_train...,OK,Archivo encontrado.
7,Existe archivo results/splits/regression_test_...,OK,Archivo encontrado.
8,Existe archivo results/metrics/classification_...,OK,Archivo encontrado.
9,Existe modelo models/trained_models/classifica...,OK,Modelo serializado encontrado.


Comprobaciones pendientes de revisión: 0


## 15. Cierre

Se entrenaron y guardaron los modelos supervisados base:

### Clasificación
- `LogisticRegression`
- `DecisionTreeClassifier`
- `SVM`

### Regresión
- `LinearRegression`
- `DecisionTreeRegressor`

Productos generados:

- métricas base en `results/metrics/`;
- particiones train/test en `results/splits/`;
- pipelines entrenados en `models/trained_models/`;
- metadata del modelado en `models/trained_models/modeling_metadata.json`.

La evaluación comparativa, matrices de confusión, curvas ROC, validación cruzada e interpretación de resultados corresponden al notebook `03_model_evaluation`.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a797108b-ff37-4667-9d30-00c3227a6ce3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>